# Uji Semua Model dengan Komentar Baru
**Skripsi: Perbandingan Algoritma XGBoost dan Model IndoBERT dalam Klasifikasi Komentar Publik**

Notebook ini memuat **semua model yang sudah dilatih** (LR, XGBoost NoTune, XGBoost Tuned, IndoBERT --
masing-masing untuk 3 skenario: A-Manual, A-Gabungan, B-Gabungan -- total **12 model**), lalu menguji
seluruhnya dengan **komentar baru** (di luar dataset) untuk melihat apakah model bisa mengklasifikasikan
ke kelas **Keluhan**, **Saran**, atau **Pujian** dengan benar.

Path pada notebook ini mengikuti `struktur_file_update.txt` (v3) yang Anda kirim:
- Model LR/XGBoost: `models/ml_v3/`
- Model IndoBERT: `models/indobert_v3/`

> ⚠️ **Yang WAJIB Anda sesuaikan**: fungsi `preprocess_xgb()` di Bagian 3. Notebook ini **tidak tahu**
> isi fungsi *preprocessing* asli Anda (`preprocessing_v2.py`), sehingga komentar baru harus melalui
> pipeline *cleaning + stemming + negation handling* **yang sama persis** dengan yang dipakai saat
> training, atau hasil prediksi model TF-IDF (LR/XGBoost) bisa tidak akurat karena kata-kata di
> komentar baru tidak cocok dengan vocabulary yang dipelajari model. IndoBERT lebih toleran karena
> tokenizer WordPiece-nya bisa memecah kata yang belum pernah dilihat.

## 1. Import Library

In [3]:
import pandas as pd
import numpy as np
import os, sys, pickle, warnings
warnings.filterwarnings('ignore')

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

pd.set_option('display.max_colwidth', 60)
print('✅ Library siap')
print('GPU tersedia:', torch.cuda.is_available())


✅ Library siap
GPU tersedia: True


## 2. Konfigurasi Path (mengikuti struktur v3)

In [4]:
BASE_DIR       = r'C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method'
MODEL_DIR      = os.path.join(BASE_DIR, 'models', 'ml_v3')        # LR & XGBoost .pkl
MODEL_DIR_IDB  = os.path.join(BASE_DIR, 'models', 'indobert_v3')  # IndoBERT checkpoint per skenario

LABEL_MAP = {'keluhan': 0, 'saran': 1, 'pujian': 2}
INV_MAP   = {v: k for k, v in LABEL_MAP.items()}
CLASS_NAMES = ['Keluhan', 'Saran', 'Pujian']

SKENARIO = ['A_Manual', 'A_Gabungan', 'B_Gabungan']

MODEL_NAME_IDB = 'indobenchmark/indobert-base-p1'
MAX_LENGTH_IDB = 128
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Daftar model LR/XGBoost yang akan dimuat: (key_model, key_vectorizer, label tampilan)
MODEL_SKLEARN = []
for s in SKENARIO:
    MODEL_SKLEARN.append((f'LR_v2_{s}',           f'LR_v2_{s}',           f'Logistic Regression - {s.replace("_","-")}'))
for s in SKENARIO:
    MODEL_SKLEARN.append((f'XGB_v2_NoTune_{s}',   f'XGB_v2_NoTune_{s}',   f'XGBoost (NoTune) - {s.replace("_","-")}'))
for s in SKENARIO:
    MODEL_SKLEARN.append((f'XGB_v2_Tuned_{s}',    f'XGB_v2_Tuned_{s}',    f'XGBoost (Tuned) - {s.replace("_","-")}'))

print(f'Total model TF-IDF (LR/XGBoost) : {len(MODEL_SKLEARN)}')
print(f'Total model IndoBERT            : {len(SKENARIO)}')


Total model TF-IDF (LR/XGBoost) : 9
Total model IndoBERT            : 3


## 3. Fungsi Preprocessing (TF-IDF & IndoBERT)

Memakai fungsi preprocessing ASLI yang sudah diekstrak menjadi modul terpisah:
- `preprocessing_v2.py` -> fungsi `preprocess_v2()` (dipakai jalur LR/XGBoost, TF-IDF)
- `preprocessing_indobert.py` -> fungsi `preprocess_indobert()` (dipakai jalur IndoBERT)

**Sebelum menjalankan cell ini**, pastikan KEDUA file `.py` tersebut sudah disalin ke folder
`BASE_DIR`, dan folder `dictionaries/` (emoji_dict.csv, slang_dict.csv, dst.) bisa diakses dari path
yang dipakai di dalam kedua modul.

> Catatan: `preprocess_indobert()` sebelumnya TIDAK dipanggil sama sekali di notebook ini -- komentar
> baru langsung masuk ke tokenizer IndoBERT tanpa preprocessing apa pun. Ini sudah diperbaiki di
> Bagian 6 (`predict_idb`) supaya konsisten dengan pipeline yang dipakai saat training.

In [ ]:
sys.path.insert(0, BASE_DIR)

DICT_DIR = os.path.join(BASE_DIR, 'dictionaries')

# ── Jalur TF-IDF (LR/XGBoost): preprocess_v2() ───────────────────────────────
try:
    from preprocessing_v2 import preprocess_v2 as preprocess_xgb, muat_semua_dictionary
    muat_semua_dictionary(DICT_DIR)   # <-- diganti dari BASE_DIR jadi DICT_DIR
    print('✅ preprocess_xgb() -> memakai preprocess_v2() asli dari preprocessing_v2.py')
except ImportError as e:
    import re
    def preprocess_xgb(text):
        """Fallback sederhana -- HANYA dipakai kalau preprocessing_v2.py tidak ditemukan.
        TIDAK melakukan stemming/stopword removal/negation handling seperti pipeline asli."""
        text = str(text).lower()
        text = re.sub(r'http\S+|www\.\S+', ' ', text)
        text = re.sub(r'@\w+', ' ', text)
        text = re.sub(r'#\w+', ' ', text)
        text = re.sub(r'[^a-z\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    print(f'⚠️  preprocessing_v2.py tidak ditemukan ({e}) -- pakai fallback sederhana.')

# ── Jalur IndoBERT: preprocess_indobert() ────────────────────────────────────
try:
    from preprocessing_indobert import preprocess_indobert, muat_dictionary_indobert
    muat_dictionary_indobert(BASE_DIR)  # <-- ini TETAP BASE_DIR, jangan diubah
    print('✅ preprocess_indobert() siap dari preprocessing_indobert.py')
except ImportError as e:
    def preprocess_indobert(text):
        """Fallback: tanpa preprocessing sama sekali (teks mentah apa adanya)."""
        return str(text)
    print(f'⚠️  preprocessing_indobert.py tidak ditemukan ({e}) -- IndoBERT pakai teks mentah.')

Loading semua dictionary...
✅ emoji_dict     : 226 emoji individu (marah=55, senang=76, harap=95)
✅ profanity_set  : 213 kata kasar -> [BADWORD]
✅ slang_dict     : 14,689 kata slang -> baku
✅ stopwords_id   : 799 kata (negasi sudah diproteksi)
✅ kbbi_words     : 72,441 kata (termasuk negasi, domain, badword)

✅ Semua dictionary berhasil dimuat! preprocess_v2() siap dipakai.
✅ preprocess_xgb() -> memakai preprocess_v2() asli dari preprocessing_v2.py
Slang dict: 14,688 kata
Emoji dict: 144 emoji -> kata deskriptif

Semua kamus IndoBERT siap!
✅ preprocess_indobert() siap dari preprocessing_indobert.py


## 4. Muat Semua Model TF-IDF (LR & XGBoost)

In [9]:
models_sklearn = {}

for model_key, vec_key, label in MODEL_SKLEARN:
    try:
        model = pickle.load(open(os.path.join(MODEL_DIR, f'{model_key}.pkl'), 'rb'))
        vec   = pickle.load(open(os.path.join(MODEL_DIR, f'vec_{vec_key}.pkl'), 'rb'))
        models_sklearn[label] = {'model': model, 'tfidf': vec['tfidf'], 'selector': vec['selector']}
        print(f'  ✅ {label}')
    except FileNotFoundError as e:
        print(f'  ❌ {label} -- file tidak ditemukan: {e}')

print(f'\nTotal model TF-IDF berhasil dimuat: {len(models_sklearn)} / {len(MODEL_SKLEARN)}')


  ✅ Logistic Regression - A-Manual
  ✅ Logistic Regression - A-Gabungan
  ✅ Logistic Regression - B-Gabungan
  ✅ XGBoost (NoTune) - A-Manual
  ✅ XGBoost (NoTune) - A-Gabungan
  ✅ XGBoost (NoTune) - B-Gabungan
  ✅ XGBoost (Tuned) - A-Manual
  ✅ XGBoost (Tuned) - A-Gabungan
  ✅ XGBoost (Tuned) - B-Gabungan

Total model TF-IDF berhasil dimuat: 9 / 9


## 5. Muat Semua Model IndoBERT

In [10]:
models_idb = {}

for s in SKENARIO:
    label = f'IndoBERT - {s.replace("_","-")}'
    save_dir = os.path.join(MODEL_DIR_IDB, f'indobert_{s}')
    try:
        tokenizer = AutoTokenizer.from_pretrained(save_dir)
        model = AutoModelForSequenceClassification.from_pretrained(save_dir).to(DEVICE)
        model.eval()
        models_idb[label] = {'model': model, 'tokenizer': tokenizer}
        print(f'  ✅ {label}')
    except Exception as e:
        print(f'  ❌ {label} -- gagal dimuat: {e}')

print(f'\nTotal model IndoBERT berhasil dimuat: {len(models_idb)} / {len(SKENARIO)}')


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  ✅ IndoBERT - A-Manual


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  ✅ IndoBERT - A-Gabungan


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  ✅ IndoBERT - B-Gabungan

Total model IndoBERT berhasil dimuat: 3 / 3


## 6. Fungsi Prediksi Terpadu (Semua Model)

In [11]:
def predict_sklearn(text, bundle):
    clean = preprocess_xgb(text)
    X = bundle['selector'].transform(bundle['tfidf'].transform([clean]))
    pred = bundle['model'].predict(X)[0]
    proba = bundle['model'].predict_proba(X)[0]
    return INV_MAP[pred].capitalize(), float(proba[pred])

def predict_idb(text, bundle):
    tokenizer, model = bundle['tokenizer'], bundle['model']
    clean = preprocess_indobert(text)  # samakan dengan preprocessing saat training
    enc = tokenizer(clean, max_length=MAX_LENGTH_IDB, padding='max_length',
                     truncation=True, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        logits = model(**enc).logits
        proba = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    pred = int(np.argmax(proba))
    return INV_MAP[pred].capitalize(), float(proba[pred])

def predict_all_models(text):
    """Jalankan SEMUA model (12) terhadap satu komentar, kembalikan DataFrame hasil."""
    rows = []
    for label, bundle in models_sklearn.items():
        pred, conf = predict_sklearn(text, bundle)
        rows.append({'Model': label, 'Prediksi': pred, 'Confidence': f'{conf*100:.1f}%'})
    for label, bundle in models_idb.items():
        pred, conf = predict_idb(text, bundle)
        rows.append({'Model': label, 'Prediksi': pred, 'Confidence': f'{conf*100:.1f}%'})
    return pd.DataFrame(rows)

print('✅ Fungsi predict_all_models() siap dipakai')


✅ Fungsi predict_all_models() siap dipakai


## 7. Uji dengan Contoh Komentar Baru

Silakan ubah/tambah daftar `KOMENTAR_BARU` di bawah -- idealnya sertakan contoh yang jelas untuk
tiap kelas (Keluhan/Saran/Pujian) sekaligus contoh yang ambigu, agar terlihat konsistensi antar model.

In [12]:
KOMENTAR_BARU = [
    "Aplikasi BPJS sering error saat mau daftar antrian, tolong diperbaiki dong",
    "Coretax bagus tapi menurut saya perlu ditambahkan fitur notifikasi jatuh tempo pajak",
    "Terima kasih MBG, anak saya jadi semangat sekolah karena dapat makan bergizi gratis",
    "Kenapa aplikasi ini lemot banget sih, udah 3 hari gak bisa login sama sekali",
    "Semoga ke depannya bisa ditambahkan fitur live chat untuk customer service",
    "Pelayanannya cepat dan ramah, sangat membantu saya mengurus BPJS",
]

for i, komentar in enumerate(KOMENTAR_BARU, 1):
    print(f'\n{"="*90}')
    print(f'Komentar #{i}: "{komentar}"')
    print(f'{"="*90}')
    hasil = predict_all_models(komentar)
    print(hasil.to_string(index=False))



Komentar #1: "Aplikasi BPJS sering error saat mau daftar antrian, tolong diperbaiki dong"
                           Model Prediksi Confidence
  Logistic Regression - A-Manual  Keluhan      75.6%
Logistic Regression - A-Gabungan  Keluhan      66.8%
Logistic Regression - B-Gabungan  Keluhan      69.7%
     XGBoost (NoTune) - A-Manual  Keluhan      78.5%
   XGBoost (NoTune) - A-Gabungan    Saran      71.8%
   XGBoost (NoTune) - B-Gabungan    Saran      56.1%
      XGBoost (Tuned) - A-Manual  Keluhan      86.0%
    XGBoost (Tuned) - A-Gabungan  Keluhan      52.6%
    XGBoost (Tuned) - B-Gabungan  Keluhan      57.7%
             IndoBERT - A-Manual  Keluhan      95.6%
           IndoBERT - A-Gabungan  Keluhan      99.7%
           IndoBERT - B-Gabungan  Keluhan      99.7%

Komentar #2: "Coretax bagus tapi menurut saya perlu ditambahkan fitur notifikasi jatuh tempo pajak"
                           Model Prediksi Confidence
  Logistic Regression - A-Manual  Keluhan      38.6%
Logistic Regr

## 8. Ringkasan Perbandingan (Semua Komentar × Semua Model)

Tabel pivot ini memudahkan melihat apakah model-model **sepakat** atau **berbeda pendapat** terhadap
komentar yang sama -- kolom yang isinya seragam menunjukkan model konsisten, kolom yang bervariasi
menunjukkan komentar tersebut ambigu/sulit diklasifikasikan.

In [13]:
rekap = []
for komentar in KOMENTAR_BARU:
    hasil = predict_all_models(komentar)
    row = {'Komentar': komentar[:50] + ('...' if len(komentar) > 50 else '')}
    for _, r in hasil.iterrows():
        row[r['Model']] = r['Prediksi']
    rekap.append(row)

df_rekap = pd.DataFrame(rekap).set_index('Komentar')
df_rekap


,Logistic Regression - A-Manual,Logistic Regression - A-Gabungan,Logistic Regression - B-Gabungan,XGBoost (NoTune) - A-Manual,XGBoost (NoTune) - A-Gabungan,XGBoost (NoTune) - B-Gabungan,XGBoost (Tuned) - A-Manual,XGBoost (Tuned) - A-Gabungan,XGBoost (Tuned) - B-Gabungan,IndoBERT - A-Manual,IndoBERT - A-Gabungan,IndoBERT - B-Gabungan
Komentar,,,,,,,,,,,,
Aplikasi BPJS sering error saat mau daftar antrian...,Keluhan,Keluhan,Keluhan,Keluhan,Saran,Saran,Keluhan,Keluhan,Keluhan,Keluhan,Keluhan,Keluhan
Coretax bagus tapi menurut saya perlu ditambahkan ...,Keluhan,Keluhan,Keluhan,Saran,Keluhan,Keluhan,Keluhan,Keluhan,Keluhan,Saran,Saran,Saran
"Terima kasih MBG, anak saya jadi semangat sekolah ...",Pujian,Pujian,Pujian,Pujian,Pujian,Pujian,Pujian,Pujian,Pujian,Pujian,Pujian,Pujian
"Kenapa aplikasi ini lemot banget sih, udah 3 hari ...",Keluhan,Keluhan,Keluhan,Keluhan,Keluhan,Keluhan,Keluhan,Keluhan,Keluhan,Keluhan,Keluhan,Keluhan
Semoga ke depannya bisa ditambahkan fitur live cha...,Pujian,Pujian,Pujian,Pujian,Pujian,Pujian,Pujian,Pujian,Pujian,Saran,Saran,Saran
"Pelayanannya cepat dan ramah, sangat membantu saya...",Saran,Saran,Saran,Saran,Saran,Saran,Saran,Saran,Saran,Pujian,Pujian,Pujian


## 9. Mode Interaktif — Uji Komentar Anda Sendiri

Jalankan sel ini untuk mengetik komentar secara langsung dan melihat prediksi seluruh model.
Ketik `selesai` untuk berhenti.

In [15]:
while True:
    komentar = input('Masukkan komentar (atau ketik "selesai" untuk berhenti): ')
    if komentar.strip().lower() == 'selesai':
        print('Selesai.')
        break
    if not komentar.strip():
        continue
    print(f'\nKomentar: "{komentar}"')   # <-- baris baru ini
    hasil = predict_all_models(komentar)
    print(hasil.to_string(index=False))
    print('-' * 90)   # <-- opsional, pemisah antar komentar biar rapi


Komentar: "harusnya program mbg ini diberikan ke yang lebih layak "
                           Model Prediksi Confidence
  Logistic Regression - A-Manual    Saran      71.9%
Logistic Regression - A-Gabungan    Saran      52.6%
Logistic Regression - B-Gabungan    Saran      53.2%
     XGBoost (NoTune) - A-Manual    Saran      62.8%
   XGBoost (NoTune) - A-Gabungan    Saran      58.1%
   XGBoost (NoTune) - B-Gabungan    Saran      51.6%
      XGBoost (Tuned) - A-Manual    Saran      76.3%
    XGBoost (Tuned) - A-Gabungan    Saran      75.3%
    XGBoost (Tuned) - B-Gabungan    Saran      79.8%
             IndoBERT - A-Manual    Saran      99.9%
           IndoBERT - A-Gabungan    Saran     100.0%
           IndoBERT - B-Gabungan    Saran      99.9%
------------------------------------------------------------------------------------------

Komentar: "kenapa di coretax itu website nya sering gabisa masukin inputan yah"
                           Model Prediksi Confidence
  Logistic Regres